# EDM-98 Training Preparation

This notebook shows a minimal preparation flow for using `edm98` as the label and split source for EDMFormer-style training.

It focuses on the parts that matter for dataset preparation:

- loading the packaged EDM-98 metadata
- inspecting Deezer IDs and record metadata
- mapping records to externally downloaded audio
- understanding the embedding directory layout EDMFormer expects
- constructing the dataset config fields used by the EDMFormer loaders

This notebook does **not** run full training. It prepares the inputs that training expects.

## What EDMFormer Expects

From the original EDMFormer / SongFormer data loading flow, the important dataset-side requirements are:

- a label JSONL file with `id` and `labels`
- split ID files such as `train.txt` and `val.txt`
- four embedding directories:
  - `musicfm_30s`
  - `muq_30s`
  - `musicfm_420s`
  - `muq_420s`
- files named like `<id>_<start_sec>.npy`

The `edm98` package already gives you the canonical labels and split IDs. You still need to download audio separately and generate embeddings externally.

For the current EDM-98 workflow, the canonical audio contract is that each downloaded song is named as `<deezer_id>.<ext>`, such as `1060564312.mp3`.

## Prerequisites

Before using this notebook, make sure you have:

- installed `edm98`
- downloaded the EDM-98 songs separately
- stored those songs as `<deezer_id>.<ext>` in a local audio directory
- decided where your training workspace and generated embeddings will live

This notebook assumes the audio already exists locally. It does not fetch songs from Deezer for you.

In [ ]:
from pathlib import Path

from edm98.loaders import load_all_splits, load_dataset_records, load_records_by_split

In [ ]:
records = load_dataset_records()
splits = load_all_splits()

len(records), {name: len(ids) for name, ids in splits.items()}

## Inspect One Record

Each record contains:

- `id`: the Deezer ID
- `labels`: the canonical EDM-98 structure labels
- `file_path`: the original filename used during labeling, when available

The canonical lookup key for local audio, however, is `id`, with files stored as `<deezer_id>.<ext>`.

In [ ]:
records[0]

## Point To Downloaded Audio

The package does not distribute the songs themselves. Assume you have already downloaded the audio externally into a local directory.

Set `AUDIO_DIR` to your local folder and inspect which EDM-98 records can be resolved by Deezer ID.

In [ ]:
AUDIO_DIR = Path('/path/to/downloaded/audio')
EXTENSIONS = ('.mp3', '.wav', '.flac', '.m4a')

resolved = []
missing = []

for record in records:
    candidate = None
    for ext in EXTENSIONS:
        path = AUDIO_DIR / f"{record['id']}{ext}"
        if path.exists():
            candidate = path
            break
    if candidate is not None:
        resolved.append((record['id'], candidate.name))
    else:
        missing.append(record['id'])

len(resolved), len(missing)

If your local filenames do not follow the `<deezer_id>.<ext>` convention yet, rename them or build that mapping before generating embeddings. The training pipeline should ultimately resolve audio by Deezer ID.

In [ ]:
resolved[:5], missing[:5]

## Inspect Train / Val / Test Records

The packaged split files can be used directly.

In [ ]:
train_records = load_records_by_split('train')
val_records = load_records_by_split('val')
test_records = load_records_by_split('test')

len(train_records), len(val_records), len(test_records)

## Expected Embedding Layout

EDMFormer training expects four embedding directories. A typical local training workspace might look like:

```text
training_workspace/
  labels/
    dataset.jsonl
  splits/
    train.txt
    val.txt
    test.txt
  embeddings/
    musicfm_30s/
    muq_30s/
    musicfm_420s/
    muq_420s/
```

Each embedding file should be named like `<id>_<start_sec>.npy`, for example:

- `1060564312_0.npy`
- `1060564312_30.npy`
- `1060564312_60.npy`

The 30s directories correspond to 420s windows built from concatenated 30s segments, matching EDMFormer’s expected input format.

In [ ]:
WORKSPACE = Path('/path/to/training_workspace')

label_path = WORKSPACE / 'labels' / 'dataset.jsonl'
train_split_path = WORKSPACE / 'splits' / 'train.txt'
val_split_path = WORKSPACE / 'splits' / 'val.txt'

musicfm_30s = WORKSPACE / 'embeddings' / 'musicfm_30s'
muq_30s = WORKSPACE / 'embeddings' / 'muq_30s'
musicfm_420s = WORKSPACE / 'embeddings' / 'musicfm_420s'
muq_420s = WORKSPACE / 'embeddings' / 'muq_420s'

input_embedding_dir = ' '.join(
    str(path)
    for path in (musicfm_30s, muq_30s, musicfm_420s, muq_420s)
)

input_embedding_dir

## EDMFormer-Compatible Dataset Abstracts

The EDMFormer config generator ultimately fills dataset entries with the following fields:

- `internal_tmp_id`
- `dataset_type`
- `input_embedding_dir`
- `label_path`
- `split_ids_path`
- `multiplier`

A minimal EDM-98 training/eval pair looks like this:

In [ ]:
train_item = {
    'internal_tmp_id': 'EDMFormer',
    'dataset_type': 'EDMFormer',
    'input_embedding_dir': input_embedding_dir,
    'label_path': str(label_path),
    'split_ids_path': str(train_split_path),
    'multiplier': 1,
}

eval_item = {
    'internal_tmp_id': 'EDMFormer',
    'dataset_type': 'EDMFormer',
    'input_embedding_dir': input_embedding_dir,
    'label_path': str(label_path),
    'split_ids_path': str(val_split_path),
    'multiplier': 1,
}

train_item, eval_item

## Optional: Export Packaged EDM-98 Assets Into A Training Workspace

If you want a self-contained local workspace, you can copy the packaged dataset and split files into a `training_workspace/` folder before generating embeddings.

In [ ]:
from importlib import resources
import shutil

workspace_labels = WORKSPACE / 'labels'
workspace_splits = WORKSPACE / 'splits'
workspace_labels.mkdir(parents=True, exist_ok=True)
workspace_splits.mkdir(parents=True, exist_ok=True)

dataset_resource = resources.files('edm98.resources').joinpath('dataset.jsonl')
shutil.copyfile(dataset_resource, workspace_labels / 'dataset.jsonl')

for split_name in ('train', 'val', 'test'):
    split_resource = resources.files('edm98.resources.splits').joinpath(f'{split_name}.txt')
    shutil.copyfile(split_resource, workspace_splits / f'{split_name}.txt')

sorted(str(path.relative_to(WORKSPACE)) for path in WORKSPACE.rglob('*') if path.is_file())[:10]

## Next Step

After the labels, split files, audio, and embeddings are ready, point EDMFormer at:

- `label_path`
- `train_split_ids_path`
- `eval_split_ids_path`
- `input_embedding_dir`

That is the minimal contract required for EDMFormer-style training with EDM-98.